# 02. Build the Map

This notebook reads the small files made by `01datapreparation.ipynb` and draws the Version 1 web app: one map showing how busy each Rapid Rail station is, and what kind of place it serves.

| Section | What it does |
|---|---|
| 0 | **Visual settings: the place to experiment** (shapes, colours, sizes, lines, text) |
| 1 | Load the prepared files |
| 2 | Base map and boundaries |
| 3 | Station markers |
| 4 | Click-to-show destinations |
| 5 | Title, legend and notes |
| 6 | Save the web app |

## 0. Visual settings

Colours can be any CSS colour: a name (`"red"`, `"teal"`) or a hex code (`"#1f77b4"`). A colour picker such as [htmlcolorcodes.com](https://htmlcolorcodes.com) gives hex codes.

In [235]:
# ---------------- Station markers ----------------
MARKER_SHAPE     = "square"    # <- "circle", "square" or "diamond"
MARKER_MIN_SIZE  = 6           # <- width in pixels of the quietest station
MARKER_MAX_SIZE  = 40          # <- width in pixels of the busiest station (Bukit Bintang)
MARKER_OPACITY   = 1        # <- 0 = invisible, 1 = solid
MARKER_BORDER_COLOUR = "#333333" # <- outline around each marker
MARKER_BORDER_WIDTH  = 1.5       # <- outline thickness in pixels (0 = no outline)

# ---------------- Marker colours: weekend ratio ----------------
# weekend_ratio = weekend and holiday entries / working-day entries
COLOUR_COMMUTER = "#0213ff"    # <- stations that empty at weekends (ratio around 0.3)
COLOUR_MIXED    = "#df90ff"    # <- stations in between (ratio around 0.75)
COLOUR_WEEKEND  = "#ff0b0b"    # <- stations busier at weekends (ratio 1.2 or more)
RATIO_COMMUTER  = 0.3          # <- ratio at which COLOUR_COMMUTER is reached
RATIO_MIXED     = 0.75         # <- ratio shown in COLOUR_MIXED
RATIO_WEEKEND   = 1.2          # <- ratio at which COLOUR_WEEKEND is reached

# ---------------- Destination lines (shown when a station is clicked) ----------------
LINE_COLOUR     = "#fbff00"    # <- colour of the lines to the top destinations
LINE_MIN_WIDTH  = 4            # <- thinnest line in pixels
LINE_MAX_WIDTH  = 12           # <- thickest line in pixels (the busiest destination overall)
LINE_OPACITY    = 0.9          # <- 0 = invisible, 1 = solid
LINE_DASHED     = False        # <- True for dashed lines
HIGHLIGHT_COLOUR = "#fbff00"   # <- ring drawn around the clicked station

# ---------------- Boundaries ----------------
DISTRICT_LINE_COLOUR = "#444444"   # <- district outlines
DISTRICT_LINE_WIDTH  = 2           # <- thickness in pixels
SHOW_DISTRICTS_AT_START = False  # <- True to show district outlines when the page opens
CONSTITUENCY_LINE_COLOUR = "#000000"  # <- constituency outlines
CONSTITUENCY_LINE_WIDTH  = 1          # <- thickness in pixels
SHOW_CONSTITUENCIES_AT_START = False  # <- True to show constituency outlines when the page opens

# ---------------- Base map and text ----------------
BASEMAP     = "OpenStreetMap"  # <- background map; "OpenStreetMap" needs no key
START_ZOOM  = 12               # <- higher = closer in; 11 shows the whole Klang Valley
MAP_CENTRE  = [3.08, 101.65]   # <- [latitude, longitude] the map opens on
FONT        = "Segoe UI, Helvetica, Arial, sans-serif"  # <- font for the title and panels
TITLE       = "MALAYSIA (KLANG VALLEY): RAPID RAIL RIDES" # <- main heading
SUBTITLE    = ("Rapid rail entries by station - how busy is each station?")
PANEL_BACKGROUND = "rgba(255, 255, 255, 0.92)"  # <- background of the title, legend and info panels

## 1. Load the prepared files

In [236]:
from pathlib import Path
import json, math
import pandas as pd
import geopandas as gpd
import folium
from branca.colormap import LinearColormap
from branca.element import MacroElement, Template, Element

PROCESSED = Path("data/processed")
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

stations = gpd.read_file(PROCESSED / "stations.geojson")
flows = pd.read_csv(PROCESSED / "flows_top.csv")
districts = gpd.read_file(PROCESSED / "districts.geojson")
constituencies = gpd.read_file(PROCESSED / "constituencies.geojson")
meta = json.loads((PROCESSED / "metadata.json").read_text())

stations = stations.sort_values("avg_daily_entries", ascending=False).reset_index(drop=True)
print(f"{len(stations)} stations, {len(flows)} flows, period {meta['period_start']} to {meta['period_end']}")

154 stations, 770 flows, period 2026-06-01 to 2026-08-31


## 2. Base map and boundaries

In [237]:
# VISUAL: centre, zoom, background
m = folium.Map(location=MAP_CENTRE, zoom_start=START_ZOOM, tiles=BASEMAP, control_scale=True, prefer_canvas=False)
m.get_root().header.add_child(Element("<style>.leaflet-tile-pane { filter: grayscale(70%); opacity: 0.8; }</style>"))  # VISUAL: grey, faded background (remove this line for full colour)

# VISUAL: district outline colour
# VISUAL: district outline width
folium.GeoJson(
    districts, name="Districts", show=SHOW_DISTRICTS_AT_START,
    style_function=lambda f: {"color": DISTRICT_LINE_COLOUR, "weight": DISTRICT_LINE_WIDTH, "fillOpacity": 0},
    tooltip=folium.GeoJsonTooltip(["district"], labels=False),
).add_to(m)

# VISUAL: constituency outline colour
# VISUAL: constituency outline width
# VISUAL: "4 4" = dash pattern
folium.GeoJson(
    constituencies, name="Parliamentary constituencies", show=SHOW_CONSTITUENCIES_AT_START,
    style_function=lambda f: {"color": CONSTITUENCY_LINE_COLOUR, "weight": CONSTITUENCY_LINE_WIDTH, "dashArray": "4 4", "fillOpacity": 0},           
    tooltip=folium.GeoJsonTooltip(["parlimen"], labels=False),
).add_to(m)

## 3. Station markers

Marker **area** grows with average daily entries (the width grows with the square root), so a station with four times the entries looks four times as big, not sixteen. Colour shows the weekend ratio. Busier stations are drawn underneath quieter ones so small stations stay visible.

In [238]:
# VISUAL: the three colours
colour_scale = LinearColormap(
    colors=[COLOUR_COMMUTER, COLOUR_MIXED, COLOUR_WEEKEND],
    index=[RATIO_COMMUTER, RATIO_MIXED, RATIO_WEEKEND],
    vmin=RATIO_COMMUTER, vmax=RATIO_WEEKEND)

max_entries = stations["avg_daily_entries"].max()

def marker_size(entries):
    """Width in pixels: area proportional to entries."""
    return MARKER_MIN_SIZE + (MARKER_MAX_SIZE - MARKER_MIN_SIZE) * math.sqrt(entries / max_entries)

def marker_colour(ratio):
    return colour_scale(min(max(ratio, RATIO_COMMUTER), RATIO_WEEKEND))   # values beyond the ends use the end colour

def marker_html(size, colour):
    """The marker is a small box drawn with CSS; its shape comes from MARKER_SHAPE."""
    shape_css = {
        "circle":  "border-radius: 50%;",                  # VISUAL: round corners all the way = circle
        "square":  "border-radius: 2px;",                  # VISUAL: slightly rounded square corners
        "diamond": "border-radius: 2px; transform: rotate(45deg) scale(0.8);",  # VISUAL: square turned 45 degrees
    }[MARKER_SHAPE]
    return (f'<div style="width:{size:.0f}px; height:{size:.0f}px; background:{colour}; '
            f'opacity:{MARKER_OPACITY}; border:{MARKER_BORDER_WIDTH}px solid {MARKER_BORDER_COLOUR}; '
            f'box-sizing:border-box; box-shadow:0 0 2px rgba(0,0,0,0.5); cursor:pointer; {shape_css}"></div>')

def tooltip_html(s):
    return (f"<b>{s['name']}</b> <span style='color:#777'>({s['codes']})</span><br>"
            f"{s['avg_daily_entries']:,.0f} entries a day<br>"
            f"Working days: {s['avg_working_day']:,.0f} &middot; Weekends and holidays: {s['avg_weekend_holiday']:,.0f}<br>"
            f"Weekend ratio: {s['weekend_ratio']:.2f}<br>"
            f"<span style='color:#777'>{s['district']}</span><br>"
            f"<i style='color:#777'>Click to see top destinations</i>")

station_layer = folium.FeatureGroup(name="Stations").add_to(m)
marker_names = {}   # station key -> the marker's name in the web page, used in Section 4

for rank, s in stations.iterrows():
    size = marker_size(s["avg_daily_entries"])
    mk = folium.Marker(
        location=[s["lat"], s["lon"]],
        icon=folium.DivIcon(html=marker_html(size, marker_colour(s["weekend_ratio"])), icon_size=(size, size), icon_anchor=(size / 2, size / 2)),
        tooltip=folium.Tooltip(tooltip_html(s), sticky=True),
        z_index_offset=rank * 10,          # busier stations first = underneath
    ).add_to(station_layer)
    marker_names[s["key"]] = mk.get_name()

print(f"Drew {len(marker_names)} markers as {MARKER_SHAPE}s, {MARKER_MIN_SIZE} to {MARKER_MAX_SIZE} px")

Drew 154 markers as squares, 6 to 40 px


## 4. Click-to-show destinations

Clicking a station draws lines to its top destinations and lists them in a panel. Clicking the map background clears them. Folium cannot do this on its own, so this cell adds a short piece of JavaScript. **You do not need to edit it**: its colours and widths come from Section 0.

In [239]:
# Data handed to the web page
station_info = {s["key"]: {"name": s["name"], "lat": s["lat"], "lon": s["lon"], "entries": s["avg_daily_entries"], "size": marker_size(s["avg_daily_entries"])} for _, s in stations.iterrows()}
flow_info = {k: g[["dest_key", "avg_daily_trips", "share_of_origin"]].values.tolist() for k, g in flows.sort_values("rank").groupby("origin_key")}
style = {"lineColour": LINE_COLOUR, "lineMin": LINE_MIN_WIDTH, "lineMax": LINE_MAX_WIDTH,
         "lineOpacity": LINE_OPACITY, "dashed": LINE_DASHED, "highlight": HIGHLIGHT_COLOUR,
         "maxTrips": float(flows["avg_daily_trips"].max()), "topN": int(meta["top_n"])}

CLICK_JS = r"""
(function() {
  var map = {{ this._parent.get_name() }};
  var stations = __STATIONS__;
  var flows = __FLOWS__;
  var style = __STYLE__;
  var markers = {__MARKERS__};
  var layer = L.layerGroup().addTo(map);
  var panel = document.getElementById("flow-panel");
  var fmt = function(n) { return Math.round(n).toLocaleString("en-GB"); };

  function clearFlows() {
    layer.clearLayers();
    panel.innerHTML = "<i>Click a station to see where its passengers go.</i>";
  }
  function showFlows(key) {
    layer.clearLayers();
    var o = stations[key];
    var rows = "";
    (flows[key] || []).forEach(function(f, i) {
      var d = stations[f[0]];
      if (!d) { return; }
      var w = style.lineMin + (style.lineMax - style.lineMin) * Math.sqrt(f[1] / style.maxTrips);
      L.polyline([[o.lat, o.lon], [d.lat, d.lon]], {
        color: style.lineColour, weight: w, opacity: style.lineOpacity,
        dashArray: style.dashed ? "8 6" : null, interactive: false
      }).addTo(layer);
      rows += "<tr><td>" + (i + 1) + ".</td><td>" + d.name + "</td><td style='text-align:right'>" + fmt(f[1]) +
              "</td><td style='text-align:right;color:#777'>" + Math.round(f[2] * 100) + "%</td></tr>";
    });
    L.circleMarker([o.lat, o.lon], {radius: o.size / 2 + 5, color: style.highlight, weight: 3,
                                    fill: false, interactive: false}).addTo(layer);
    panel.innerHTML = "<b>" + o.name + "</b>: top " + style.topN + " destinations<br>" +
      "<span style='color:#777'>" + fmt(o.entries) + " entries a day</span>" +
      "<table style='margin-top:6px;border-collapse:collapse'><tr style='color:#777'><td></td><td>Destination</td>" +
      "<td style='text-align:right;padding-left:8px'>Trips/day</td><td style='text-align:right;padding-left:8px'>Share</td></tr>" +
      rows + "</table>";
  }
  Object.keys(markers).forEach(function(key) {
    markers[key].on("click", function(e) { L.DomEvent.stopPropagation(e); showFlows(key); });
  });
  map.on("click", clearFlows);
  clearFlows();
})();
"""

js = (CLICK_JS.replace("__STATIONS__", json.dumps(station_info))
              .replace("__FLOWS__", json.dumps(flow_info))
              .replace("__STYLE__", json.dumps(style))
              .replace("__MARKERS__", ", ".join(f'"{k}": {v}' for k, v in marker_names.items())))
click_js = MacroElement()
click_js._template = Template("{% macro script(this, kwargs) %}" + js + "{% endmacro %}")
m.add_child(click_js)
print(f"Click behaviour added for {len(flow_info)} stations")

Click behaviour added for 154 stations


## 5. Title, legend and notes

In [240]:
panel_css = (f"position:absolute; z-index:1000; background:{PANEL_BACKGROUND}; font-family:{FONT}; "   # VISUAL: panel look
             "padding:10px 14px; border-radius:6px; box-shadow:0 1px 5px rgba(0,0,0,0.3); font-size:14px;")

period = f"{pd.Timestamp(meta['period_start']):%d %b %Y} to {pd.Timestamp(meta['period_end']):%d %b %Y}"

# Title panel, top left
title_html = f"""
<div style="{panel_css} top:14px; left:60px; max-width:600px;">
  <div style="font-size:20px; font-weight:600;">{TITLE}</div>
  <div style="color:#555; margin-top:2px;">{SUBTITLE}</div>
  <div style="color:#777; margin-top:4px; font-size:12px;">Average per day, {period}</div>
</div>"""

# Destinations panel, top right (below the layer switcher)
flow_panel_html = f'<div id="flow-panel" style="{panel_css} top:60px; right:12px; width:320px;"></div>'

# Legend, bottom left: size examples and colour bar
examples = [1000, 10000, int(round(max_entries, -3))]
size_rows = "".join(
    f'<div style="display:flex; align-items:center; margin:3px 0;">'
    f'<div style="width:{MARKER_MAX_SIZE + 4}px; display:flex; justify-content:center;">{marker_html(marker_size(n), "#bbbbbb")}</div>'
    f'<span style="margin-left:6px;">{n:,} entries a day</span></div>' for n in examples)
gradient = f"linear-gradient(to right, {COLOUR_COMMUTER}, {COLOUR_MIXED}, {COLOUR_WEEKEND})"
legend_html = f"""
<div style="{panel_css} bottom:50px; left:14px; width:300px;">
  <div style="font-weight:600; margin-bottom:4px;">Station size: level of activity</div>
  {size_rows}
  <div style="font-weight:600; margin:8px 0 4px;">Colour: weekend ratio</div>
  <div style="height:12px; border-radius:3px; background:{gradient}; border:1px solid #ccc;"></div>
  <div style="display:flex; justify-content:space-between; color:#555; font-size:12px; margin-top:2px;">
    <span>Commuter<br>{RATIO_COMMUTER}</span><span style="text-align:center">Mixed<br>{RATIO_MIXED}</span><span style="text-align:right">Weekend<br>{RATIO_WEEKEND}+</span>
  </div>
  <div style="color:#777; font-size:12px; margin-top:6px;">Ratio = weekend and public holiday entries divided by working-day entries.</div>
</div>"""

# Notes, bottom right: data note and sources
notes_html = f"""
<div style="{panel_css} bottom:30px; right:12px; width:300px; font-size:12px; color:#555;">
  <b>About the data</b><br>
  Level of activity = average daily entries (tap-ins). Exits at 32 Putrajaya Line stations are missing in the source data, so activity uses entries only and no destination line ends at those stations.<br><br>
  Sources: data.gov.my<br>
  (Prasarana ridership and GTFS, CC BY 4.0); DOSM boundaries; map &copy; OpenStreetMap contributors.<br>
  Prepared {meta['prepared_on']}.
</div>"""

for html in (title_html, flow_panel_html, legend_html, notes_html):
    m.get_root().html.add_child(Element(html))

folium.LayerControl(collapsed=True).add_to(m)

## 6. Save the web app

In [241]:
out_file = OUTPUT / "kl_mobility_v1.html"
m.get_root().header.add_child(Element("<title>MALAYSIA(KLANG VALLEY): RAPID RAIL RIDES</title>"))
m.save(out_file)
print(f"Saved {out_file} ({out_file.stat().st_size / 1e3:.0f} KB). Open it in a browser for the full-size view.")
m

Saved output\kl_mobility_v1.html (348 KB). Open it in a browser for the full-size view.


http://localhost:8000/index.html
___
**END**